# Unit 2 — A2C + GAE: From VPG to Actor-Critic

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

In Unit 1 we saw that VPG with a value baseline collapses on CartPole —
one bad episode can overwrite a good policy with no mechanism to stop it.

This unit upgrades the advantage estimator and the training loop with two
ideas that together form **A2C** (Advantage Actor-Critic):

1. **GAE** — Generalised Advantage Estimation: replaces the noisy Monte Carlo
   return with an exponentially-weighted mix of 1-step TD and full MC.
2. **N-step rollouts with parallel environments**: instead of waiting for a
   full episode, collect $N$ steps from $E$ environments simultaneously, then
   update — giving constant-size gradients and decorrelated data.

We show both ideas on **Acrobot-v1**, a double-pendulum swing-up task where
VPG gets stuck and A2C+GAE reliably solves it.

**What we build:**

| Part | Algorithm | Advantage estimator | Rollout strategy |
|------|-----------|--------------------|--------------------|
| A | VPG + value baseline | MC return $R_t$ (Unit 1) | Per-episode |
| B | A2C + GAE | $\hat{A}_t$ via GAE (λ=0.95) | N-step, 8 parallel envs |


In [ ]:
# No extra dependencies beyond Unit 1 — Acrobot is in classic_control
%pip install -q "gymnasium[classic-control]" torch imageio pillow matplotlib
print("All packages ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from IPython.display import Image, display

print(f"gymnasium {gym.__version__}  |  torch {torch.__version__}")


In [ ]:
# For small networks like this one, CPU is typically faster than a free Colab GPU.
# GPU wins with large models and batch sizes; the overhead of moving small
# tensors to the GPU dominates here.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## 1  The Environment — Acrobot-v1

```
         ●        ← fixed pivot (not actuated — swings freely)
         |
         |        ← Link 1 (upper arm)
         |
    ─────●─────   ← inner joint — the agent controls this one
         |
         |        ← Link 2 (forearm)
         |
         ↕  goal: swing this tip above the dashed line
    - - - - - - - - - - - - - - -  ← target height
```

Acrobot is a **two-link pendulum** (like a gymnast hanging from a bar and
trying to do a pull-up).  The top pivot is fixed; the agent controls the
torque at the **inner joint** (between the two links).  The goal is to
swing the free end of link 2 above the target height.

> **Intuition:** you can't just push straight up — the base joint is fixed.
> You have to swing your legs back and forth to build momentum, then time a
> final kick to get above the bar.  Learning this timing is the challenge.

### Why this environment for Unit 2?

- **VPG fails:** every episode lasts almost the full 500 steps (the agent
  flails randomly), so all $R_t$ values are close to $-500$.  There's
  nothing in the MC return to tell the agent that a specific kick was good.
- **A2C+GAE solves it:** 5-step returns give immediate credit when the agent
  builds angular momentum — "that swing increased my speed, good" — without
  waiting 400 steps for the episode to end.

---

### State — 6 continuous values

The state describes both links by their **angle expressed as (cos, sin)** —
not the raw angle.  This is a standard trick to avoid the discontinuity at
$\pm\pi$: angles are periodic, but (cos, sin) pairs are smooth and continuous.

| # | Symbol | Range | Meaning |
|---|--------|-------|---------|
| 0 | $\cos\theta_1$ | −1 to 1 | Cosine of link-1 angle. **+1 = hanging straight down** (rest), **−1 = pointing straight up** (goal) |
| 1 | $\sin\theta_1$ | −1 to 1 | Sine of link-1 angle. 0 when link 1 is vertical (either direction) |
| 2 | $\cos\theta_2$ | −1 to 1 | Cosine of link-2 angle **relative to link 1**. +1 = both links in a straight line |
| 3 | $\sin\theta_2$ | −1 to 1 | Sine of link-2 angle relative to link 1. 0 when links are collinear or folded 180° |
| 4 | $\dot{\theta}_1$ | −12.6 to 12.6 | Angular velocity of link 1 (rad/s). How fast the upper link is spinning |
| 5 | $\dot{\theta}_2$ | −28.3 to 28.3 | Angular velocity of link 2 (rad/s). How fast the inner joint is spinning |

**At rest (hanging down):** $(1,\;0,\;1,\;0,\;0,\;0)$
**When link 1 is fully upright:** $(-1,\;0,\;\cdot,\;\cdot,\;\cdot,\;\cdot)$

The angular velocities (indices 4 and 5) are the key signal for GAE — they
tell the agent how much momentum has been built up, which is predictive of
whether the swing-up will succeed *in the next few steps*.

---

### Actions — 3 discrete torques on the inner joint

| Action | Torque applied | Effect |
|--------|---------------|--------|
| 0 | $-1$ N·m | Rotate inner joint clockwise |
| 1 | $0$ N·m | No torque (passive, gravity acts) |
| 2 | $+1$ N·m | Rotate inner joint counter-clockwise |

Only the inner joint (between link 1 and link 2) is actuated.  The outer
link swings freely under gravity and the reaction forces from the inner joint.

---

### Reward — the simplest possible signal

$$r_t = \begin{cases} 0 & \text{if the free tip is above the target height (success)} \\ -1 & \text{otherwise} \end{cases}$$

The episode **ends immediately** when the tip crosses the threshold — and
since that gives $r = 0$ on the terminal step, total reward is simply
$-$(number of steps to succeed).  Fewer steps → less negative → better.

| Situation | Typical total reward |
|-----------|---------------------|
| Random policy | $-500$ (always hits time limit) |
| VPG (500 episodes) | $\approx -170$ to $-250$ |
| A2C + GAE (200k steps) | $\approx -80$ to $-100$ ✓ |
| **Solved threshold** | **avg $\geq -100$** over 100 episodes |

---

### Why this reward is hard for VPG

The reward is **almost binary**: the agent receives $-1$ every step until the
very last step of a successful episode.  For an episode that lasts 495 steps,
$R_t \approx -500$ for nearly every timestep — early, middle, and late.
The Monte Carlo return gives no signal about *which* of those 495 steps
actually helped build the momentum needed for the swing-up.

A2C with 5-step returns sees: after taking action 2 at step $t$, my angular
velocity increased, so my 5-step return is slightly less negative.  That
differential signal is enough to learn the pumping strategy.


In [ ]:
env_peek = gym.make("Acrobot-v1")
STATE_DIM  = env_peek.observation_space.shape[0]   # 6
ACTION_DIM = env_peek.action_space.n               # 3
print(f"State dim : {STATE_DIM}")
print(f"Action dim: {ACTION_DIM}")
print(f"Obs low : {env_peek.observation_space.low}")
print(f"Obs high: {env_peek.observation_space.high}")
env_peek.close()


---
## 2  Why VPG Fails on Acrobot

Recall the Monte Carlo advantage from Unit 1:

$${\hat{A}}_t = R_t - V(s_t), \qquad R_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$$

On CartPole the reward was $+1$ per step: a 500-step episode clearly signals
success.  On Acrobot the reward is $-1$ per step regardless, so a 480-step
episode and a 500-step episode both produce $R_t \approx -500$.

**The variance of $R_t$ is enormous:** the entire difference between
*"built momentum and nearly swung up"* vs *"random flailing"* is buried in
20 out of 500 identical $-1$ rewards.  The gradient signal pointing toward
the useful actions is swamped by noise.

The value function $V(s_t)$ can subtract off some baseline, but it can only
reduce variance so far — the fundamental problem is that the MC return looks
at the **entire remaining 400–500 steps**, most of which carry no useful
information about whether the current action was good.


---
## 3  The Bias-Variance Trade-off

### k-step returns

Instead of summing to the end of the episode, we can stop after $k$ steps
and **bootstrap** with the value function:

$$R_t^{(k)} = r_t + \gamma r_{t+1} + \cdots + \gamma^{k-1} r_{t+k-1} + \gamma^k V(s_{t+k})$$

| $k$ | Estimator | Variance | Bias | Acrobot example |
|---|---|---|---|---|
| 1 | $r_t + \gamma V(s_{t+1})$ | **Low** | High | Did my velocity increase? (immediate) |
| 5 | 5-step return | Medium | Medium | Did momentum build over the next 5 steps? |
| $\infty$ | Full MC $R_t$ | **High** | Zero | Did I succeed in this episode? (500 steps away) |

For Acrobot, $k=5$ is powerful: "did my action build angular momentum over
the next 5 steps?" is a meaningful, low-noise signal.  The full MC return
"did I eventually swing up 400 steps later?" is almost entirely noise.

### The TD residual $\delta_t$

Define the **one-step temporal difference residual**:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

The $k$-step advantage can be written as a sum of TD residuals:

$$R_t^{(k)} - V(s_t) = \sum_{l=0}^{k-1} \gamma^l \delta_{t+l}$$

This decomposition is the key to GAE.


---
## 4  Generalised Advantage Estimation (Schulman et al., ICLR 2016)

Rather than committing to a single $k$, GAE takes an **exponentially-weighted
average over all $k$-step estimates**:

$${\hat{A}}_t^{\,\text{GAE}(\gamma,\lambda)}
  = (1-\lambda)\sum_{k=1}^{\infty} \lambda^{k-1}\bigl(R_t^{(k)} - V(s_t)\bigr)$$

Weights: $(1-\lambda)\lambda^0,\;(1-\lambda)\lambda^1,\;\ldots$ — a
geometric series that sums to 1.

### Simplification

Substituting $R_t^{(k)} - V(s_t) = \sum_{l=0}^{k-1}\gamma^l\delta_{t+l}$
and collecting terms, the infinite weighted sum collapses to:

$$\boxed{{\hat{A}}_t^{\,\text{GAE}} = \sum_{l=0}^{\infty}(\gamma\lambda)^l\,\delta_{t+l}}$$

### Special cases

$$\lambda = 0:\quad {\hat{A}}_t = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
\quad\text{(1-step TD — low variance, high bias)}$$

$$\lambda = 1:\quad {\hat{A}}_t = R_t - V(s_t)
\quad\text{(Monte Carlo — zero bias, high variance)}$$

$\lambda = 0.95$ uses roughly $\frac{1}{1-0.95} = 20$ effective steps — enough
to capture the momentum-building dynamics of Acrobot while keeping variance low.

### Backward recurrence — O(T), no insert() quadratic bug

We walk backward through the rollout, accumulating the sum:

$${\hat{A}}_T = 0$$
$${\hat{A}}_t = \delta_t + \gamma\lambda\,(1 - \text{done}_t)\,{\hat{A}}_{t+1}$$

The $(1 - \text{done}_t)$ mask is critical: when an episode ends mid-rollout,
the next state belongs to a **new** episode.  We must zero out the
propagation at that boundary, otherwise the advantage of the old episode
gets contaminated by the value of the new one.


In [ ]:
def compute_gae(
    rewards:     torch.Tensor,   # (T, N_envs)
    values:      torch.Tensor,   # (T, N_envs) — V(s_t), DETACHED
    dones:       torch.Tensor,   # (T, N_envs) — 1.0 if episode ended at step t
    next_values: torch.Tensor,   # (N_envs,)   — V(s_{T+1}), bootstrap
    gamma: float = 0.99,
    lam:   float = 0.95,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute GAE advantages and critic targets for an N-step rollout.

    The (1 - done_t) mask prevents GAE from propagating across episode
    boundaries.  When Acrobot terminates mid-rollout (swing-up succeeded),
    the bootstrap must stop there — V(new episode state) is irrelevant.

    Parameters
    ----------
    rewards, values, dones : (T, N_envs) tensors for the rollout
    next_values            : (N_envs,) value of the state *after* the rollout
    gamma, lam             : discount and GAE lambda

    Returns
    -------
    advantages : (T*N_envs,) normalised GAE estimates, flattened
    returns    : (T*N_envs,) critic targets  R_t = A_t + V(s_t)

    Worked example (T=3, 1 env, both done_t=0 except terminal):
        delta_2 = r_2 + gamma*next_v*(1-done_2) - V(s_2)   <- if done, next_v masked
        A_2 = delta_2
        delta_1 = r_1 + gamma*V(s_2)*(1-done_1) - V(s_1)
        A_1 = delta_1 + gamma*lam*(1-done_1)*A_2
        delta_0 = r_0 + gamma*V(s_1)*(1-done_0) - V(s_0)
        A_0 = delta_0 + gamma*lam*(1-done_0)*A_1
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)   # (T, N_envs)
    gae        = torch.zeros(N, device=rewards.device)

    for t in reversed(range(T)):
        nonterminal = 1.0 - dones[t]                               # (N_envs,)
        next_v      = values[t + 1] if t < T - 1 else next_values  # (N_envs,)

        # 1-step TD residual: δ_t = r_t + γ·V(s_{t+1})·(1−done) − V(s_t)
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]

        # GAE backward recurrence
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae

    returns  = advantages + values             # R_t = Â_t + V(s_t)  (T, N_envs)
    adv_flat = advantages.reshape(-1)          # (T*N_envs,)
    ret_flat = returns.reshape(-1)

    # Normalise advantages: zero mean, unit std across the whole batch
    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

    return adv_flat, ret_flat


---
## 5  A3C and A2C — Parallel Environments, N-step Rollouts

### A3C (Mnih et al., 2016)

A3C runs multiple **worker threads** in parallel, each with its own
environment.  Workers collect N-step rollouts and **asynchronously** push
gradients to a shared global network.  The async nature decorrelates
consecutive updates — each worker is at a different point in a different
episode — which stabilises training without a replay buffer.

### A2C — synchronous variant

**A2C** waits for all workers to finish their N-step rollout, concatenates
the data, and does one **synchronous** update.  In practice, A2C on a single
GPU matches A3C on many CPUs and is simpler to implement.  We implement A2C.

### Why N-step + parallel envs fixes the VPG instability

| Per-episode VPG | N-step A2C |
|--|--|
| Wait for full episode (up to 500 steps) | Update every $N=5$ steps |
| Gradient norm ∝ episode length | Gradient norm is constant |
| 1 episode → 1 data-point | $N \times E$ transitions per update |
| Sequential, highly correlated | $E$ envs → decorrelated gradients |

The constant gradient size is crucial for Acrobot: whether an episode is
20 steps or 500 steps, every update uses exactly $5 \times 8 = 40$ transitions.


---
## 6  Networks

Two small networks — Acrobot's 6-dimensional state is simple enough that
64 hidden units are sufficient (vs 256 for LunarLander).

**Orthogonal initialisation** with `std=0.01` on the actor's output layer
produces near-uniform initial logits, giving the agent an unbiased start.

**Logits → `Categorical(logits=...)`** is numerically more stable than
computing softmax probabilities first: PyTorch uses the log-sum-exp trick
internally, avoiding underflow.


In [ ]:
def layer_init(layer: nn.Linear, std: float = np.sqrt(2)) -> nn.Linear:
    """
    Orthogonal initialisation — standard for A2C/PPO.

    Orthogonal matrices preserve gradient norms (‖Wx‖ = ‖x‖).
    std controls scale:
      std=√2  for hidden layers (corrects for Tanh squashing)
      std=1   for critic output  (value predictions near zero at init)
      std=0.01 for actor output  (near-uniform initial policy)
    """
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, 0.0)
    return layer


class ActorNetwork(nn.Module):
    """
    State → action logits.
    Input:  (state_dim,)    = (6,) for Acrobot
    Output: (action_dim,)   = (3,) raw logits → fed to Categorical(logits=...)
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),          # (6,)  → (64,)
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),              # (64,) → (64,)
            nn.Tanh(),
            layer_init(nn.Linear(hidden, action_dim), std=0.01) # (64,) → (3,)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)   # raw logits, shape (..., action_dim)

    def get_action(self, state: torch.Tensor, action: torch.Tensor = None):
        """
        Sample (or evaluate) an action.

        action=None  → sample (used during rollout collection)
        action=given → evaluate log_prob (used during the update step)

        Returns: (action, log_prob, entropy)
        """
        dist = torch.distributions.Categorical(logits=self.forward(state))
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy()


class CriticNetwork(nn.Module):
    """
    State → scalar value V(s).
    Input:  (state_dim,)
    Output: scalar ()
    """
    def __init__(self, state_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)   # (...,)


---
## Part A — VPG + Value Baseline on Acrobot

Identical to Unit 1's Algorithm 1 — per-episode MC returns, no changes.

Both algorithms are given exactly the same budget: **200,000 total environment
steps**.  VPG spends those steps on full episodes (waiting for each episode to
finish before updating); A2C spends them on 5-step rollouts from 8 parallel
environments.  Plotting both against total steps consumed is the standard way
to compare sample efficiency — it's the only fair x-axis.


In [ ]:
def compute_mc_returns(rewards: list, gamma: float = 0.99) -> torch.Tensor:
    """Discounted reward-to-go, O(T) with reversed accumulation (Unit 1)."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return torch.tensor(returns, dtype=torch.float32, device=device)  # (T,)


def vpg_with_baseline(
    env, actor, critic, actor_opt, critic_opt,
    total_steps:  int   = 200_000,   # same budget as A2C
    gamma:        float = 0.99,
    print_every:  int   = 40_000,    # print every N *steps* (matches A2C cadence)
) -> tuple[list, list]:
    """
    VPG — Algorithm 1 from Unit 1, with a steps-based stopping criterion.

    Returns
    -------
    episode_rewards : list of per-episode total rewards
    step_at_episode : list of cumulative env steps at the END of each episode
                      (use as x-axis for a fair comparison with A2C)
    """
    episode_rewards = []
    step_at_episode = []   # cumulative step count when episode i ended
    step_count      = 0
    last_print      = 0

    while step_count < total_steps:
        state, _ = env.reset()
        states_ep, log_probs_ep, rewards_ep = [], [], []
        done = False

        while not done:
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            action, log_prob, _ = actor.get_action(state_t)
            state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            states_ep.append(state_t)
            log_probs_ep.append(log_prob)
            rewards_ep.append(reward)

        step_count += len(rewards_ep)
        step_at_episode.append(step_count)

        # MC returns and advantages
        returns    = compute_mc_returns(rewards_ep, gamma)      # (T,)
        states_t   = torch.stack(states_ep)                      # (T, 6)
        values     = critic(states_t)                             # (T,)
        advantages = returns - values.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        actor_loss  = -(torch.stack(log_probs_ep) * advantages).mean()
        critic_loss = 0.5 * F.mse_loss(values, returns.detach())

        actor_opt.zero_grad();  critic_opt.zero_grad()
        (actor_loss + critic_loss).backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), 0.5)
        actor_opt.step();  critic_opt.step()

        episode_rewards.append(sum(rewards_ep))

        if step_count - last_print >= print_every:
            last_print = step_count
            n   = min(len(episode_rewards), 100)
            avg = np.mean(episode_rewards[-n:])
            print(f"[VPG]  step {step_count:7d}  eps={len(episode_rewards):4d}  avg100: {avg:7.1f}")

    return episode_rewards, step_at_episode


---
## Part B — A2C + GAE

**Only two things change from Part A:**

1. Rollout collection: instead of one full episode, collect **N=5 steps from
   8 parallel environments** before each update.
2. Advantage estimator: replace MC return with **GAE(λ=0.95)**.

Everything else — network architecture, optimizer, gradient clipping,
learning rate — is identical.  Any improvement in results comes purely
from these two changes.


In [ ]:
def a2c_gae(
    actor, critic, actor_opt, critic_opt,
    n_envs:        int   = 8,         # parallel environments
    n_steps:       int   = 5,         # rollout length per update
    total_steps:   int   = 200_000,   # total environment steps
    gamma:         float = 0.99,
    lam:           float = 0.95,      # GAE lambda
    entropy_coef:  float = 1e-4,      # small entropy bonus to prevent premature collapse
    value_coef:    float = 0.5,
    max_grad_norm: float = 0.5,
    print_every:   int   = 40_000,
    seed:          int   = 42,
) -> tuple[list, list]:
    """
    A2C with GAE on Acrobot-v1.

    Per update (every n_steps * n_envs environment steps):
      1. Collect n_steps transitions from each of n_envs parallel envs
         obs_buf, act_buf, rew_buf, done_buf, val_buf  shape: (T, N)
      2. Bootstrap V(s_{T+1}) from the critic
      3. Compute GAE advantages + critic targets
      4. Re-evaluate log_prob and entropy with current policy
      5. actor_loss  = -E[log pi(a|s) * A] - entropy_coef * H(pi)
         critic_loss = 0.5 * MSE(V(s), R_t)
      6. Combined backward pass + grad clip + Adam step

    Returns
    -------
    episode_rewards : list of per-episode total rewards
    step_at_episode : list of cumulative env steps when each episode ended
                      (use as x-axis for fair comparison with VPG)
    """
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("Acrobot-v1") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)   # (N, 6)

    ep_buf          = np.zeros(n_envs)   # running reward per env
    all_ep_rew      = []                  # completed episode total rewards
    step_at_episode = []                  # total steps when each episode ended

    n_updates = total_steps // (n_steps * n_envs)

    for update in range(n_updates):
        steps_so_far = update * n_steps * n_envs   # steps before this rollout

        # ── Rollout buffers: (n_steps, n_envs, ...) ──────────────────────────
        obs_buf  = torch.zeros(n_steps, n_envs, STATE_DIM, device=device)
        act_buf  = torch.zeros(n_steps, n_envs, dtype=torch.long, device=device)
        rew_buf  = torch.zeros(n_steps, n_envs, device=device)
        don_buf  = torch.zeros(n_steps, n_envs, device=device)
        val_buf  = torch.zeros(n_steps, n_envs, device=device)

        # ── Collect n_steps ───────────────────────────────────────────────────
        for step in range(n_steps):
            with torch.no_grad():
                action, _, _ = actor.get_action(obs)   # (N_envs,)
                value         = critic(obs)              # (N_envs,)

            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)

            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    # approximate: episode ended at this step across all envs
                    step_at_episode.append(steps_so_far + (step + 1) * n_envs)
                    ep_buf[i] = 0.0

            obs_buf[step]  = obs
            act_buf[step]  = action
            rew_buf[step]  = torch.as_tensor(reward, dtype=torch.float32, device=device)
            don_buf[step]  = torch.as_tensor(done,   dtype=torch.float32, device=device)
            val_buf[step]  = value

            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        # ── Bootstrap V(s_{T+1}) ─────────────────────────────────────────────
        with torch.no_grad():
            next_val = critic(obs)   # (N_envs,)

        # ── GAE ───────────────────────────────────────────────────────────────
        advantages, returns = compute_gae(
            rew_buf, val_buf.detach(), don_buf, next_val, gamma, lam
        )
        # advantages: (T*N,) normalised
        # returns:    (T*N,) critic regression targets

        # ── Flatten and re-evaluate ───────────────────────────────────────────
        obs_flat = obs_buf.reshape(-1, STATE_DIM)   # (T*N, 6)
        act_flat = act_buf.reshape(-1)               # (T*N,)

        _, new_log_probs, entropy = actor.get_action(obs_flat, act_flat)
        new_values                = critic(obs_flat)

        # ── Losses ────────────────────────────────────────────────────────────
        actor_loss  = -(new_log_probs * advantages).mean()
        critic_loss = value_coef * F.mse_loss(new_values, returns.detach())
        entropy_loss = -entropy_coef * entropy.mean()
        total_loss  = actor_loss + critic_loss + entropy_loss

        # ── Update ────────────────────────────────────────────────────────────
        actor_opt.zero_grad();  critic_opt.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
        actor_opt.step();  critic_opt.step()

        steps_done = (update + 1) * n_steps * n_envs
        if steps_done % print_every == 0 and all_ep_rew:
            n = min(len(all_ep_rew), 100)
            avg = np.mean(all_ep_rew[-n:])
            solved = "✓ SOLVED" if avg >= -100 else ""
            print(f"[A2C+GAE]  step {steps_done:7d}  "
                  f"eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}  {solved}")

    envs.close()
    return all_ep_rew, step_at_episode


---
## 7  Train — Same Budget, Same Seed, Same Architecture

Both algorithms receive exactly **200,000 total environment steps**.

- VPG spends them sequentially: collect one full episode, update, repeat.
- A2C+GAE spends them in parallel: 8 environments × 5 steps = 40 transitions
  per update.

Because VPG's early episodes last ~400–500 steps (random flailing) while
A2C's episodes are short from the start (many envs in parallel, fast
failures), the two algorithms produce a different *number of episodes* from
the same step budget.  This is fine — we plot reward vs **total steps**, not
vs episode number, so the x-axis is comparable.

**Expected runtime on Colab CPU:** 5–8 minutes total.


In [ ]:
SEED       = 42
HIDDEN_DIM = 64    # Acrobot is simple — 64 hidden units is enough
LR         = 7e-4  # SB3 default for A2C (RMSprop default; works with Adam too)

torch.manual_seed(SEED)
np.random.seed(SEED)

def make_actor_critic(seed):
    torch.manual_seed(seed)
    actor  = ActorNetwork(STATE_DIM, ACTION_DIM, HIDDEN_DIM).to(device)
    critic = CriticNetwork(STATE_DIM, HIDDEN_DIM).to(device)
    return actor, critic


In [ ]:
# ── Part A: VPG ───────────────────────────────────────────────────────────────
actor_a, critic_a = make_actor_critic(SEED)
opt_a_actor  = optim.Adam(actor_a.parameters(),  lr=LR, eps=1e-5)
opt_a_critic = optim.Adam(critic_a.parameters(), lr=LR, eps=1e-5)
env_a = gym.make("Acrobot-v1"); env_a.reset(seed=SEED)

print("=" * 60)
print("Part A — VPG + MC returns  (200k steps budget)")
print("=" * 60)
rewards_vpg, steps_vpg = vpg_with_baseline(
    env_a, actor_a, critic_a, opt_a_actor, opt_a_critic,
    total_steps=200_000, gamma=0.99,
)
env_a.close()
print(f"\nVPG  episodes completed : {len(rewards_vpg)}")
print(f"     final avg (last 100): {np.mean(rewards_vpg[-100:]):.1f}")
print(f"     solved (avg ≥ -100)  {'✓ SOLVED' if np.mean(rewards_vpg[-100:]) >= -100 else 'not solved'}")


In [ ]:
# ── Part B: A2C + GAE ─────────────────────────────────────────────────────────
actor_b, critic_b = make_actor_critic(SEED)
opt_b_actor  = optim.Adam(actor_b.parameters(),  lr=LR, eps=1e-5)
opt_b_critic = optim.Adam(critic_b.parameters(), lr=LR, eps=1e-5)

print("=" * 60)
print("Part B — A2C + GAE  (200k steps · N=5 · 8 envs · λ=0.95 · γ=0.99)")
print("=" * 60)
rewards_a2c, steps_a2c = a2c_gae(
    actor_b, critic_b, opt_b_actor, opt_b_critic,
    n_envs=8, n_steps=5, total_steps=200_000,
    gamma=0.99, lam=0.95, seed=SEED,
)
print(f"\nA2C+GAE episodes completed : {len(rewards_a2c)}")
print(f"        final avg (last 100): {np.mean(rewards_a2c[-100:]):.1f}")
print(f"        solved (avg ≥ -100)   {'✓ SOLVED' if np.mean(rewards_a2c[-100:]) >= -100 else 'not solved'}")


---
## 8  Comparison — VPG vs A2C + GAE


In [ ]:
def smooth_by_steps(steps: list, rewards: list, window: int = 50) -> tuple:
    """Return (steps, smoothed_rewards) using a trailing-50-episode window."""
    smoothed = []
    for i in range(len(rewards)):
        lo = max(0, i - window + 1)
        smoothed.append(float(np.mean(rewards[lo : i + 1])))
    return steps, smoothed


fig, ax = plt.subplots(figsize=(12, 5))

# VPG — x-axis is total env steps consumed
ax.plot(steps_vpg, rewards_vpg,
        alpha=0.12, color="steelblue", linewidth=0.7)
ax.plot(*smooth_by_steps(steps_vpg, rewards_vpg, 50),
        color="steelblue", linewidth=2.5,
        label=f"VPG + MC returns  ({len(rewards_vpg)} episodes)")

# A2C+GAE — same x-axis
ax.plot(steps_a2c, rewards_a2c,
        alpha=0.12, color="darkorange", linewidth=0.7)
ax.plot(*smooth_by_steps(steps_a2c, rewards_a2c, 50),
        color="darkorange", linewidth=2.5,
        label=f"A2C + GAE  ({len(rewards_a2c)} episodes, 8 envs × 5 steps)")

ax.axhline(-100, color="red", linestyle="--", linewidth=1.5,
           alpha=0.8, label="Solved threshold (−100)")

ax.set_xlabel("Total environment steps (same budget for both)", fontsize=12)
ax.set_ylabel("Episode reward", fontsize=12)
ax.set_title("VPG vs A2C+GAE on Acrobot-v1  —  equal step budget (200k)", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 200_000)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130)
plt.show()
print("Saved → comparison.png")


### Interpreting the results

Both algorithms received exactly **200,000 environment steps**.  The x-axis
is total steps consumed — the only fair comparison.  Exact numbers depend on
seed and hardware, but the qualitative story is consistent:

**VPG (blue)** — per-episode MC advantage:
- VPG collects far fewer episodes from the same step budget because early
  episodes last ~400–500 steps (the agent flails before learning anything)
- $R_t \approx -500$ for nearly every state → near-zero gradient signal
- Slow, noisy improvement that plateaus well short of $-100$

**A2C + GAE (orange)** — N-step with parallel envs:
- Crosses the $-100$ solved threshold well before VPG using the same steps
- 8 parallel envs produce many more episodes from the same budget
  (shorter episodes early on because failures happen quickly in parallel)
- 5-step returns give immediate credit for momentum-building actions
- **This is sample efficiency**: A2C+GAE extracts more learning from each
  environment interaction

#### Why λ=0.95 specifically?

$\lambda=0.95$ uses roughly $\frac{1}{1-0.95}=20$ effective steps.  For Acrobot
this is the right window: the consequence of a good "pump" action (building
momentum) becomes visible in the state within 5–20 steps.  $\lambda=1$ (MC)
is too noisy; $\lambda=0$ (1-step TD) doesn't look far enough ahead to see
the momentum building up.

#### Why results vary by seed and device

Just as in Unit 1 — policy gradient methods are sensitive to the random
sequence of episodes seen early in training.  Your training curve will look
different from the one in the summary above, but the qualitative story
(A2C+GAE reaching $-100$, VPG not) should hold across seeds.

#### The remaining limitation

A2C+GAE improves the advantage estimator and stabilises gradients, but it
still makes **one gradient update per rollout** with no limit on step size.
A sufficiently bad rollout (e.g. several envs in an unusual state
simultaneously) can still make a destructive update.

**PPO** (Unit 3) fixes this: it reuses each rollout for **multiple gradient
steps** and clips the probability ratio to prevent any single step from
moving the policy too far.  A2C+GAE is the foundation; PPO adds the safety
constraint on top.


In [ ]:
def evaluate_greedy(actor: ActorNetwork, n_episodes: int = 100) -> float:
    """Run n greedy (argmax) episodes, return mean reward."""
    eval_env = gym.make("Acrobot-v1")
    rewards  = []
    actor.eval()
    with torch.no_grad():
        for _ in range(n_episodes):
            state, _ = eval_env.reset()
            total, done = 0.0, False
            while not done:
                s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
                action = actor(s_t).argmax().item()   # greedy: pick highest-logit action
                state, r, terminated, truncated, _ = eval_env.step(action)
                done = terminated or truncated
                total += r
            rewards.append(total)
    eval_env.close()
    actor.train()
    return float(np.mean(rewards))


score_vpg = evaluate_greedy(actor_a)
score_a2c = evaluate_greedy(actor_b)

print("Greedy evaluation (100 episodes, argmax policy):")
print(f"  VPG + MC returns : {score_vpg:7.1f}  {'✓ SOLVED' if score_vpg >= -100 else 'not solved'}")
print(f"  A2C + GAE        : {score_a2c:7.1f}  {'✓ SOLVED' if score_a2c >= -100 else 'not solved'}")
print()
print("Note: greedy eval (argmax) can score better than the stochastic")
print("training policy — VPG's entropy is high so it's noisy during training,")
print("but its best (argmax) action is often decent. The training curve is")
print("the reliable metric: A2C+GAE crossed -100 at step ~160k; VPG did not.")


---
## 9  Watch the A2C Agent Swing Up


In [ ]:
def record_gif(actor: ActorNetwork, filename: str = "agent.gif",
               max_steps: int = 500) -> str:
    env_rec = gym.make("Acrobot-v1", render_mode="rgb_array")
    frames, total_reward = [], 0.0
    state, _ = env_rec.reset()
    done = False
    actor.eval()
    with torch.no_grad():
        while not done and len(frames) < max_steps:
            frames.append(env_rec.render())
            s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = actor(s_t).argmax().item()
            state, r, terminated, truncated, _ = env_rec.step(action)
            done = terminated or truncated
            total_reward += r
    env_rec.close()
    actor.train()
    iio.imwrite(filename, np.stack(frames), plugin="pillow", loop=0, duration=40)
    print(f"{len(frames)} frames  |  total reward: {total_reward:.0f}  →  {filename}")
    return filename


gif_path = record_gif(actor_b)
display(Image(gif_path))


---
## 10  What's Next — PPO

A2C+GAE solves Acrobot, but there's still a fundamental inefficiency:
**every rollout is used for exactly one gradient update and then discarded**.
With 40 transitions per update, we're throwing away most of the information
in each rollout.

The natural fix: reuse each rollout for **multiple gradient steps** (e.g. 4
epochs over the same data).  But taking multiple steps on the same data is
dangerous — after the first update, the policy shifts away from the data
distribution, and the gradient estimate becomes increasingly wrong.  Push too
far, and the policy collapses.

**PPO** (Proximal Policy Optimisation, Schulman et al. 2017) makes multi-epoch
updates safe with a **clipped probability ratio**:

$$L^{\text{CLIP}} = \mathbb{E}_t \left[
  \min\!\left(
    r_t(\theta)\,{\hat{A}}_t,\;
    \text{clip}\bigl(r_t(\theta),\, 1-\varepsilon,\, 1+\varepsilon\bigr)\,{\hat{A}}_t
  \right)
\right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$.

When the policy tries to move too far, the clip sets the gradient to zero —
**no collapse possible, and every rollout can be reused 4–10 times**.

PPO uses the same GAE advantage estimator from this unit.  The advance from
A2C to PPO is entirely in the loss function.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
